<a href="https://colab.research.google.com/github/kelvinleonardos/classify/blob/master/Classify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Classify
Project ini merupakan bagian dari aplikasi Classify. Program yang tersaji digunakan untuk melatih model dari dataset yang telah kami buat melalui pengumpulan sampel wajah dari beberapa anggoota kelompok kama.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##1. Import Library yang akan digunakan
Kode di bawah mengimpor TensorFlow dan Keras untuk membangun dan melatih model neural network, serta ImageDataGenerator untuk augmentasi gambar. matplotlib.pyplot digunakan untuk visualisasi, dan os untuk operasi sistem file.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import layers, models
import matplotlib.pyplot as plt
import os
import numpy as np
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize


##2. Mendefinisikan Lokasi Dataset
Dataset yang telah kami buat, kami simpan di Google Drive yang kami definisikan lokasinya pada kode dibawah.

In [ ]:
train_dir = '/content/drive/MyDrive/dataset/train'
validation_dir = '/content/drive/MyDrive/dataset/validation'

##3. Augmentasi dan Pengkondisian Gambar
Kode ini membuat dua objek ImageDataGenerator. train_datagen digunakan untuk augmentasi data selama pelatihan dengan rescaling dan berbagai transformasi seperti rotasi, pergeseran, shearing, zoom, dan flipping untuk memperluas dataset pelatihan. validation_datagen hanya melakukan rescaling pada data validasi untuk menormalkan nilai piksel gambar.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)

##4. Membuat Generator Data
Kode ini membuat generator data untuk pelatihan dan validasi dengan menggunakan direktori gambar sebagai sumber data. train_generator menghasilkan batch data augmented dari train_dir, dan validation_generator menghasilkan batch data dari validation_dir, dengan ukuran batch 10, target ukuran gambar 150x150, dan mode klasifikasi kategori.

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=10,  # Kurangi batch size
    class_mode='categorical'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=10,  # Kurangi batch size
    class_mode='categorical'
)

Found 90 images belonging to 1 classes.
Found 90 images belonging to 1 classes.


##5. Membuat Arsitektur CNN
Model ini terdiri dari beberapa lapisan berurutan: dimulai dengan lapisan konvolusi 2D dengan 32 filter berukuran 3x3 untuk menangkap fitur dari input gambar berukuran 150x150x3, diikuti dengan lapisan max-pooling 2x2 untuk mengurangi dimensi. Lapisan konvolusi berikutnya menggunakan 64 filter 3x3, diikuti lagi dengan lapisan max-pooling 2x2. Selanjutnya, ada dua lapisan konvolusi berturut-turut masing-masing dengan 128 filter 3x3, yang juga diikuti oleh lapisan max-pooling 2x2 setelah masing-masing konvolusi. Setelah lapisan konvolusi dan pooling ini, output diratakan (flatten) menjadi satu dimensi dan diteruskan ke lapisan fully connected dengan 512 neuron yang menggunakan aktivasi ReLU. Akhirnya, terdapat lapisan output dengan jumlah neuron yang sama dengan jumlah kelas dalam data pelatihan (len(train_generator.class_indices)), menggunakan aktivasi softmax untuk klasifikasi multi-kelas.

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(512, activation='relu'),

    layers.Dense(len(train_generator.class_indices), activation='softmax')  # Disesuaikan untuk multi-class
])

##6. Compile Model dengan Optimizer Adam dan Categorical Crossentropy Loss
Model ini telah dikompilasi dengan menggunakan optimizer Adam dan categorical crossentropy loss. Categorical crossentropy loss, digunakan dalam klasifikasi multi-class, digunakan untuk mengukur seberapa baik model memprediksi distribusi probabilitas dari label yang benar. Metrik yang dievaluasi adalah akurasi, yang memberikan persentase prediksi yang benar dari total prediksi model. Akurasi memberikan gambaran langsung tentang performa model dalam mengklasifikasikan data yang tidak terlihat sebelumnya.

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

##7. Melatih Model
Kode ini melatih model menggunakan metode fit dari Keras, dengan data dari train_generator untuk pelatihan dan validation_generator untuk validasi. Parameter steps_per_epoch menentukan jumlah batch yang akan dieksekusi pada setiap epoch selama pelatihan, sedangkan validation_steps menentukan jumlah batch yang akan dieksekusi pada setiap epoch selama validasi. Model akan dilatih selama 30 epoch, dan hasil dari pelatihan akan disimpan dalam variabel history untuk analisis lebih lanjut.

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=14,  # Sesuaikan dengan jumlah batch yang ada
    epochs=30,
    validation_data=validation_generator,
    validation_steps=9  # Sesuaikan dengan jumlah batch yang ada
)

Epoch 1/30
14/14 ━━━━━━━━━━━━━━━━━━━━ 11s 557ms/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/30
 9/14 ━━━━━━━━━━━━━━━━━━━━ 2s 445ms/step - accuracy: 1.0000 - loss: 0.0000e+00

AttributeError: 'NoneType' object has no attribute 'items'

##8. Menyimpan Model

In [ ]:
model.save('face_classification_model.keras')

##9. Visualisasi dan Validasi Hasil Pemodelan

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend(loc='upper right')

plt.show()

# Evaluasi akhir menggunakan data validation untuk mendapatkan nilai loss dan accuracy
final_loss, final_accuracy = model.evaluate(validation_generator)
print(f'Final Loss: {final_loss}, Final Accuracy: {final_accuracy}')

# Mengkonversi model ke format TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Menyimpan model TensorFlow Lite ke dalam file
with open('face_classification_model.tflite', 'wb') as f:
    f.write(tflite_model)


##9. Kalkulasi ROC dan AUC

In [ ]:
y_true = []
y_pred = []

validation_generator.reset()
for i in range(len(validation_generator)):
    x, y = validation_generator.next()
    y_true.extend(y)
    y_pred.extend(model.predict(x))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

y_true = label_binarize(y_true, classes=[i for i in range(len(train_generator.class_indices))])
n_classes = y_true.shape[1]

fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true[:, i], y_pred[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

##10. Plot Kurva ROC

In [ ]:
plt.figure(figsize=(8, 6))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f'ROC curve (class {i}) (area = {roc_auc[i]:0.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc='lower right')
plt.show()